# **Описание проекта**

Позади много уроков и заданий, и закрепить свои знания, как обычно, вы сможете в самостоятельном проекте. Это непростой проект, в котором от вас потребуется больше самостоятельности. Этапы работы описаны кратко, и вам понадобится декомпозировать задачи, то есть разделить их на более мелкие. Опирайтесь на знания об этапах анализа данных и машинного обучения из прошлых курсов.

Перейдём к задаче. HR-аналитики компании «Работа с заботой» помогают бизнесу оптимизировать управление персоналом: бизнес предоставляет данные, а аналитики предлагают, как избежать финансовых потерь и оттока сотрудников. В этом HR-аналитикам пригодится машинное обучение, с помощью которого получится быстрее и точнее отвечать на вопросы бизнеса.

Компания предоставила данные с характеристиками сотрудников компании. Среди них — уровень удовлетворённости сотрудника работой в компании. Эту информацию получили из форм обратной связи: сотрудники заполняют тест-опросник, и по его результатам рассчитывается доля их удовлетворённости от 0 до 1, где 0 — совершенно неудовлетворён, 1 — полностью удовлетворён. 

Собирать данные такими опросниками не так легко: компания большая, и всех сотрудников надо сначала оповестить об опросе, а затем проследить, что все его прошли. 

У вас будет несколько задач. Первая — построить модель, которая сможет предсказать уровень удовлетворённости сотрудника на основе данных заказчика.

Почему бизнесу это важно: удовлетворённость работой напрямую влияет на отток сотрудников. А предсказание оттока — одна из важнейших задач HR-аналитиков. Внезапные увольнения несут в себе риски для компании, особенно если уходит важный сотрудник.
Ваша вторая задача — построить модель, которая сможет на основе данных заказчика предсказать то, что сотрудник уволится из компании.

Теперь расскажем подробнее о задачах.

**Задача 1: предсказание уровня удовлетворённости сотрудника**

Для этой задачи заказчик предоставил данные с признаками:
- id — уникальный идентификатор сотрудника;
- dept — отдел, в котором работает сотрудник;
- level — уровень занимаемой должности;
- workload — уровень загруженности сотрудника;
- employment_years — длительность работы в компании (в годах);
- last_year_promo — показывает, было ли повышение за последний год;
- last_year_violations — показывает, нарушал ли сотрудник трудовой договор за последний год;
- supervisor_evaluation — оценка качества работы сотрудника, которую дал руководитель;
- salary — ежемесячная зарплата сотрудника;
- job_satisfaction_rate — уровень удовлетворённости сотрудника работой в компании, целевой признак.

**Задача 2: предсказание увольнения сотрудника из компании**

Для этой задачи вы можете использовать те же входные признаки, что и в предыдущей задаче. Однако целевой признак отличается: это quit — увольнение сотрудника из компании.

**План исследования:**

- *Задача 1:*
1. Выполнить загрузку, и ознакомиться с данными;
2. Провести предобработку данных;
3. Выполнить исследовательский анализ;
4. Провести корреляционный анализ;
5. Ввести использование пайплайнов и обучить модели;
6. Провести анализ важности признаков;
7. Подготовить вывод исследования.

- *Задача 2:*
1. Выполнить загрузку, и ознакомиться с данными;
2. Провести предобработку данных;
3. Выполнить исследовательский анализ;
4. Добавить новый признак;
5. Провести корреляционный анализ;
6. Ввести использование пайплайнов и обучить модели;
7. Провести анализ важности признаков;
8. Подготовить вывод исследования.

- Подготовить общий вывод всего проекта.

## **0. Инициализация необходимых библиотек и методов**

In [7]:
pip install -U -q scikit-learn cufflinks phik shap seaborn optuna-integration

Note: you may need to restart the kernel to use updated packages.


In [8]:
try:
    from ydata_profiling import ProfileReport
except:
    !pip install -U -q Pillow # Для решения ошибки с профайлером
    !pip install -U -q ydata-profiling
    from ydata_profiling import ProfileReport

In [9]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:70% !important; }</style>"))

C:\Users\Eugene\AppData\Local\Temp\ipykernel_15028\49583723.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [10]:
# Импорт основных библиотек
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import numpy as np

# Импорт графической части plotly
import plotly.graph_objects as go
import plotly.express as px
from plotly.offline import plot
from plotly.subplots import make_subplots


# Импорт машинной части
import shap
# Импорт библиотек необходимых на моделей регрессии

from warnings import simplefilter # Для игнорирования предупреждений о изменении некоторых моделей в следующих версия
from scipy import stats as st
from phik import phik_matrix
from phik.report import plot_correlation_matrix
from sklearn.model_selection import train_test_split

# загружаем класс pipeline
from sklearn.pipeline import Pipeline

# загружаем классы для подготовки данных
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.compose import ColumnTransformer


# загружаем класс для работы с пропусками
from sklearn.impute import SimpleImputer

# загружаем функцию для работы с метриками
from sklearn.metrics import roc_auc_score
from sklearn.metrics import make_scorer

# импортируем класс RandomizedSearchCV и OptunaSearchCV
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from optuna.integration import OptunaSearchCV
from optuna import distributions

# загружаем нужные модели
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.svm import SVR, SVC
from sklearn.tree import plot_tree

# ttest для проверки гипотезы
from scipy import stats as st

# Для проверки адекватности модели
from sklearn.dummy import DummyRegressor, DummyClassifier
# загружаем методы отбора лучших признаков
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

simplefilter(action='ignore', category=FutureWarning) # Включаем фильтр предупреждений

In [11]:
# Задаём основные показатели размерности выборок и порядка разделения на выборки
RANDOM_STATE = 50

# **Задание 1: предсказание уровня удовлетворённости сотрудника**

## **1. Загрузка данных и знакомство с данными**

### 1.1. Загрузка и изучение тренировочных данных

In [15]:
train_data = pd.read_csv('https://code.s3.yandex.net/datasets/train_job_satisfaction_rate.csv', index_col='id') # Загрузка тренировочных данных

In [ ]:
ProfileReport(train_data, title="Profiling Report") # Получение основной информации о сведениях в тренировочных данных

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 9/9 [00:00<00:00, 95.21it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
train_data.info() # Общая проверка

#### Выводы после знакомства с тренировочными данными
- В данных присутствуют как численные, так и категориальные признаки. При этом их можно дополнительно разделить на бинарные/категориальные/категориально-ранговые;
- В данных есть суммарно 10 пропусков в столбцах dept и level;
- Некоторые столбцы обладают высоким уровне корреляции. Например, должность с зарплатой и опытом работы. Нужно провести дополнительный анализ на наличие значимой мультиколлинеарности;
- Наблюдается диссбаланс классов во всех категориальных признаков. Сильнейшим образом это проявлется в столбце last_year_promo;
- В столбце level наблюдается опечатка в данных;
- Явных и неявных дубликатов в данных не наблюдается.

### 1.2. Загрузка и изучение тестовых данных

In [ ]:
test_data = pd.read_csv('https://code.s3.yandex.net/datasets/test_features.csv',index_col='id') # Загрузка тестовой выборки

In [ ]:
y_test_data = pd.read_csv('https://code.s3.yandex.net/datasets/test_target_job_satisfaction_rate.csv',index_col='id') # Загрузка тестовых сведений целевого признака

In [ ]:
ProfileReport(test_data, title="Profiling Report") # Получение основной информации о сведениях в тренировочных данных

In [ ]:
ProfileReport(y_test_data, title='Profiling Report')

#### Выводы после знакомства с тестовыми данными
- Выводы можно совместить с выводами после знакомства с тренировочными данными;
- В таблице присутствует малое число пропусков;
- Присутствует опечатка;
- Также наблюдает корреляция между некоторыми признаками;
- Тестотыв данных достаточно много. Можно разбить их на 2 выборки - валидационную и тестовую.

### Общие выводы после знакомства с данными
- В данных есть несколько пропусков. Заполним их средним значением в пайплайне;
- Также присутствуют пропуски в виде пустой строки. Их лучше перевести в NaN;
- Наименование столбцов соответствует всем требованиям. Дополнительные изменения не нужны;
- Явных и неявных дубликатов в данных нет;
- Есть опечатка, которая не повлияет на результат работы модели, но лучше её убрать;
- Наблюдает корреляция между некоторыми признаками. Нужно проверить её значимость, дабы избежать мультиколлинеарности;
- Размер тестовой выборке равен 50% в сравнении с тренировочными данными. Можно выделить из тестовых данных валидационную выборкуж
- Столбец id можно первести в index датафреймов.

## **2. Предобработка данных**

In [ ]:
train_data['level'] = train_data['level'].str.replace('sinior','senior') # Избавление от опечатки

In [ ]:
train_data.drop_duplicates(inplace=True)

In [ ]:
test_data['level'] = test_data['level'].str.replace('sinior','senior') # Избавление от опечатки в тестовых данных

In [ ]:
test_data['workload'] = test_data['workload'].replace(' ', np.nan) # Избавимся от неизвестной нагрузки
test_data['dept'] = test_data['dept'].replace(' ', np.nan)

In [ ]:
full_test_data = test_data.merge(y_test_data, left_index=True, right_index=True) # Объединим данные для удобства проверки в дальнейшем анализе
full_test_data.info()

### Выводы после предобработки данных
- Избавились от опечатки в данных;
- Перевели столбец id в индекст датафреймов;
- Избавились от пустых значений. Перевели их в nan;
- Разделили тестовые данные на валидационную и тестовую выборки;

Данные готовы к дальнейшему исследовательскому анализу.

## **3. Исследовательский анализ данных**

### 3.1. Реализация вспомогательных функций визуализации

In [ ]:
# Функция построения гистограммы данных с учётом влияния дополнительного признака
def draw_hist_columns(data, columns, color=''):
    for column in columns:
        if column in data.columns:
            if color != '':
                fig = px.histogram(data, x=column, color=color, title='Распределение столбца '+column + ' с оценкой влияния ' + color, histnorm='density')
            else:
                fig = px.histogram(data, x=column, title='Распределение столбца '+column)
            fig.show()

In [ ]:
# Функция построения гистограммы данных с учётом влияния дополнительного признака
def draw_count_columns(data, columns, color=''):
    for column in columns:
        if column in data.columns:
            if color != '':
                fig = px.histogram(data, x=column, color=color, title='Распределение столбца '+column + ' с оценкой влияния ' + color, barmode='group')
            else:
                fig = px.histogram(data, x=column, title='Распределение столбца '+column)
            fig.show()

In [ ]:

def draw_pie_columns(data, columns, color=''):
    for column in columns:
        if color == '':
            fig = px.pie(data,
                        names=column,
                        title = 'Распределение популярности значений столбца '+ column)
            fig.update_traces(textposition='inside', textinfo='percent+label')
        else:
            unique_color = data[color].unique()
            dict_specs = [{'type':'pie'} for k in range(len(unique_color))]
            fig = make_subplots(rows=1, cols=len(unique_color), specs=[dict_specs], subplot_titles=[f"{color} == {col}" for col in data[color].dropna().unique()])
            for i, c in enumerate(unique_color):
                fig.add_trace(go.Pie(values=data[data[color] == c][column].dropna().value_counts(),
                                 labels=data[data[color] == c][column].dropna().unique().tolist(),
                                # domain=dict(x=[0, 0.5]),
                                 name=""), 
                                          row=1, col=i+1)
            fig.update_layout(title_text="Круговые диаграммы показателей столбца " + column) 
        fig.show()
        
#draw_pie_columns(train_data, bin_col, 'level')

In [ ]:
# Строим boxplot для столбцов с некатегориальным распределением данных
def draw_boxplot_columns(data, columns, color=''):
    for column in columns:
        if color != '':
            fig = px.box(data, 
                     x=column, 
                     color=color,
                     title='Ящик с усами столбца '+ str(column) + 'с оценкой влияния ' + color,
                     points='all',
                    )
        else:
            fig = px.box(data, 
                     x=column,
                     title='Ящик с усами столбца '+ str(column),
                     points='all',
                    )
        fig.show()

In [ ]:
# Функция построения нормированной гистограммы данных с учётом влияния дополнительного признака
def draw_histnorm_columns(data, columns, color=''):
    for column in columns:
        if column in data.columns:
            if color != '':
                plt.figure(figsize=(13,10))
                sns.kdeplot(data=data, x=column, hue=color, multiple="stack")
                plt.title('Нормированная гистограмма распределения столбца '+ str(column) + ' с оценкой влияния ' + color)
                plt.show()
                
            else:
                fig = px.histogram(data, x=column, title='Распределение столбца '+column, histnorm='density')
                fig.show()

In [ ]:
# Разделение признаков по харастеру данных
num_col = ['salary', 'job_satisfaction_rate'] 
discret_col = ['employment_years','supervisor_evaluation']
bin_col = ['last_year_promo', 'last_year_violations']
cat_col = ['dept','level','workload']

### 3.2. Исследовательский анализ тренировочных данных

#### 3.2.1. Первичный анализ признаков

In [ ]:
draw_boxplot_columns(train_data, num_col) # Ящик с усами непрерывных признаков

In [ ]:
draw_pie_columns(train_data, cat_col) # Информация о категориальных столбцах в виде круговой диаграммы

In [ ]:
draw_pie_columns(train_data, bin_col) # Анализ распределения бинарных данных

In [ ]:
draw_count_columns(train_data, discret_col)

#### Выводы после первичного анализа признаков:
- Средняя зарплата находится на уровне 30 тысяч;
- Средняя удовлетворённость рабочим местом соответствует уровню 0.55;
- Самый популярный отдел - отдел продаж;
- Меньше всего сотрудников уровня Senior. Middle и Junior примерно в равной пропорции, но новичков больше;
- Половина всех сотрудников имеют средний уровень загруженности;
- 3% от всех сотрудников получали повышение;
- Почти 15% нарушали трудовой договор;
- Больше всего в компании молодых сотрудников, проработавших в компании не более 3х лет;
- Чаще всего сотрудники получали оценку работы 3-4;

#### 3.2.2. Анализ влияния категориальных показателей на данные

In [ ]:
for col in cat_col: # Системные обзор данных в связке с категориальными данными через соответствующие графики
    draw_pie_columns(train_data.dropna(), bin_col, col)
    draw_histnorm_columns(train_data.dropna(),num_col, col)
    draw_boxplot_columns(train_data.dropna(),num_col, col)
    draw_pie_columns(train_data.dropna(), cat_col, col)
    draw_hist_columns(train_data.dropna(), discret_col, col)

#### Выводы после оценки влияния категориальных показателей
- Больше всего сотрудников из отдела продаж;
- Сотрудников уровня junior около половины от общего числа;
- Порядка 50% сотрудников ощущают средний уровень загруженности;
- Пропорции повышения по отделам сохранены. Не наблюдается такого, что был бы перевес в каком-то отделе относительно числа сотрудников;
- Также сохранены пропорции нарушений трудового договора. Во всех отделах примерно пропорциональное число нарушений;
- Выше всего зарплаты у технологического отдела. Нижний порог у отдела продаж;
- При этом средний рейтинг удовлетворённости рабочим местом выше у тех, чья зарплата ниже - отделы продаж и закупки;
- Распределение сотрудников по отделам в целом соответствует уровню занимаемой должности. Нет пропорционального диссбаланса;
- Повышение за последний год получали лишь сотрудники уровня middle и senior. Причём в долевом соотношении сотрудники уровня senior повышение получали чаще;
- Сотрудники уровня middle и junior примерно одинаковое количество раз нарушали трудовой договор;
- Уровень занимаемой должности пропорционалез уровню заработной платы, что говорит о наличии корреляции между признаками;
- А вот уровень удовлетворённости рабочим местом в среднем ниже у профессионалов, чем у новичков;
- Выше всего уровень загруженности сотрудников в отделе HR. Проще всего живётся сотрудникам отдела sales;
- Сотрудники уровня senior в основном имеют загруженность уровня medium. В гораздо меньшей степени загруженность low;
- В большей степени высокозагруженные сотрудники получали оценку "5".

#### 3.2.3. Анализ влияния бинарных показателей на данные

In [ ]:
for col in bin_col: # Системные обзор данных в связке с бинарными данными через соответствующие графики
    draw_pie_columns(train_data.dropna(), bin_col, col)
    draw_histnorm_columns(train_data.dropna(),num_col, col)
    draw_boxplot_columns(train_data.dropna(),num_col, col)
    draw_pie_columns(train_data.dropna(), cat_col, col)
    draw_pie_columns(train_data.dropna(), discret_col, col)

#### Выводы после оценки влияния бинарных признаков

- Тех, кто получал повышение за последний год и при этом нарушал трудовой договор в процентном соотношении на треть меньше, чем тех, кто не был повышен;
- Зарплата тех, кто получал повышение в среднем выше;
- Уровень удовлетворённости у тех, кто получал повышение выше, чем у тех, кого это событие обошло стороной;
- В отделе маркетинга в процентном соотношении было повышено больше людей, чем в других отделах;
- Сотрудники уровня junior не получали повышения за последний год;
- Имеющие высокую загруженность получали повышение чаще;
- Яркой зависимости повышения от длительности работы в компании не наблюдается;
- 70% от тех, кто получил повышение за последний год получили от руководства оценку не выше 2х;
- Уровень зарплаты не влияет на наличие нарушений трудового договора;
- Те, кто не нарушал трудовой договор, как правило, имеют уровень удовлетворённости выше;
- Уровень сотрудника не связан с наличием нарушения трудового договора;
- Чаще всего трудовой договор нарушали люди умеющии низкий уровень загруженности;
- Люди имеющие нарушения трудового договора примущественно получили оценку от руководства - 4.

#### 3.2.4. Анализ влияния дискретных показателей на данные

In [ ]:
for col in discret_col: # Системные обзор данных в связке с дискретными данными через соответствующие графики
    draw_count_columns(train_data.dropna(), bin_col, col)
    draw_histnorm_columns(train_data.dropna(),num_col, col)
    draw_boxplot_columns(train_data.dropna(),num_col, col)
    draw_hist_columns(train_data.dropna(), cat_col, col)
    draw_hist_columns(train_data.dropna(), discret_col, col)

#### Выводы после оценки влияния дискретных дпоказателей на данные

- Опыт работы в компании пропорцинален уровню заработной плате;
- Уровень удовлетворённости не связан с размером зарплаты;
- Во всех отделах работаю сотрудники с разным опытом;
- Сотрудники уровня middle и senior в основонм работают в компании достаточно долго;
- Молодые сотрудники в меньшей степени нагружены работой, чем их опытные коллеги;
- Наличие нарушения трудового договора не связано с оценкой руководства;
- Уровень зарплаты не влияет на оценку руководства;
- Удовлетворённости рабочим местом выше у тех, кому руководство поставило хорошую оценку;
- Загруженность сотрудников, как их опыт работы в компании не связана с оценкой их работы;

#### Общие выводы исследовательского анализа тренировочных данных
- Больше всего сотрудников работает в отделе продаж;
- Сотрудников уровня junior около половины от общего числа;
- Порядка 50% сотрудников ощущают средний уровень загруженности;
- Выше всего зарплаты в технологическом отделе. Ниже всего в отделе продаж;
- Средний рейтинг удовлетворённости рабочим место выше у тех, кто работает в отделе с низкими зарплатами (отделы продаж и закупки);
- Сотрудники уровня junior не получали повышения за последний год;
- В средним уровень удовлетворённости ниже у профессионалов, чем у новичков;
- Самая высокая загруженность в отделе HR. Проще всего сотрудникам отдела продаж;
- Зарплата тех, кто получал повышение, в среднем выше;
- Уровень удовлетворённости рабочим местом выше у тех, кто получал повышение;
- Сильной зависимости между длительностью работы в компании и повышения не наблюдается;
- Опыт работы в компании пропорционален уровню заработной платы;
- Уровень удовлетворённости не связан с размером зарплаты;
- Молодые сотрудники в меньшей степени загружены работой;
- Наличие нарушения трудового договора не связано с оценкой руководства;
- Удовлетворённость рабочим местом выше у тех, кого начальство выше оценило;
- Загруженность сотрудников, как и их опыт работы в компании, не связана с оценкой их работы.

Многие из промежуточных выводов должны подтвердиться при проведении корреляционного анализа. Далее этим и займёмся.

### 3.3. Исследовательский анализ тестовых данных

In [ ]:
# Разделение признаков по харастеру данных
num_col = ['salary'] 
discret_col = ['employment_years','supervisor_evaluation']
bin_col = ['last_year_promo', 'last_year_violations']
cat_col = ['dept','level','workload']

#### 3.3.1. Первичный анализ признаков

In [ ]:
draw_boxplot_columns(test_data, num_col) # Ящик с усами непрерывных признаков

In [ ]:
draw_pie_columns(test_data, cat_col) # Информация о категориальных столбцах в виде круговой диаграммы

In [ ]:
draw_pie_columns(test_data, bin_col) # Анализ распределения бинарных данных

In [ ]:
draw_count_columns(test_data, discret_col)

#### Выводы после первичного анализа признаков:
- Средняя зарплата находится на уровне 30 тысяч;
- Средняя удовлетворённость рабочим местом соответствует уровню 0.55;
- Самый популярный отдел - отдел продаж;
- Меньше всего сотрудников уровня Senior. Middle и Junior примерно в равной пропорции, но новичков больше;
- Половина всех сотрудников имеют средний уровень загруженности;
- 3% от всех сотрудников получали повышение;
- Почти 15% нарушали трудовой договор;
- Больше всего в компании молодых сотрудников, проработавших в компании не более 3х лет;
- Чаще всего сотрудники получали оценку работы 3-4;

#### 3.3.2. Анализ влияния категориальных показателей на данные

In [ ]:
for col in cat_col: # Системные обзор данных в связке с категориальными данными через соответствующие графики
    draw_pie_columns(test_data.dropna(), bin_col, col)
    draw_histnorm_columns(test_data.dropna(),num_col, col)
    draw_boxplot_columns(test_data.dropna(),num_col, col)
    draw_pie_columns(test_data.dropna(), cat_col, col)
    draw_hist_columns(test_data.dropna(), discret_col, col)

#### Выводы после оценки влияния категориальных показателей
В целом, в разрезе с влиянием категориальных признаков, данные соотносятся с тренировочными. Сильных отклонений выявить не удалось.

#### 3.3.3. Анализ влияния бинарных показателей на данные

In [ ]:
for col in bin_col: # Системные обзор данных в связке с бинарными данными через соответствующие графики
    draw_pie_columns(test_data.dropna(), bin_col, col)
    draw_histnorm_columns(test_data.dropna(),num_col, col)
    draw_boxplot_columns(test_data.dropna(),num_col, col)
    draw_pie_columns(test_data.dropna(), cat_col, col)
    draw_pie_columns(test_data.dropna(), discret_col, col)

#### Выводы после оценки влияния бинарных признаков
Пропорции между тестовыми и тренировочными данными соотносятся между собой корректно. Так что все общие выводы из тренировочных данных можно продублировать и на тестовые.

#### 3.3.4. Анализ влияния дискретных показателей на данные

In [ ]:
for col in discret_col: # Системные обзор данных в связке с дискретными данными через соответствующие графики
    draw_count_columns(test_data.dropna(), bin_col, col)
    draw_histnorm_columns(test_data.dropna(),num_col, col)
    draw_boxplot_columns(test_data.dropna(),num_col, col)
    draw_hist_columns(test_data.dropna(), cat_col, col)
    draw_hist_columns(test_data.dropna(), discret_col, col)

#### Выводы после оценки влияния дискретных дпоказателей на данные
Тестовые данные в разрезе дискретных показателей корреткно соотносятся с тренировочными. Значит выводы сделанные ранее могут быть спроецированы и на тестовые данные.

#### Общие выводы исследовательского анализа тестовых данных
- Поскольку ярких отклонений в показателях тестовых данных не было обнаружено, это означает, что модели обученные на тренировочных сведениях будут корректно работать при прогнозировании результата на тестовых данных. Дальше нас ждёт корреляционных анализ данных, который должен подтвердить общие выводы исследовательского анализа касательно влияния признаков друг на друга и на целевой признак.

### Выводы исследовательского анализа данных

- Была выполненна тщательная проверка данных, как первичная, так и в разрезе влияния признаков разных типов;
- Радикальных различий между структурами тренировочных и тестовых данных не было обнаружено;
- Следующим этапом работы станет проведения корреляционного анализа с целью подтверждения общий выводов исследовательского анализа.

## **4. Корреляционный анализ данных**

In [ ]:
# Функция отрисовки диаграмм рассеяния количественных признаков в связке с оценкой влияния категориального признака
def draw_scatter_matrix(data, quant_attrib, categorical_cols):
    for col in quant_attrib:
        fig, axes = plt.subplots(1, len(categorical_cols), figsize=(30, 6))
        for i,cl in enumerate(data[categorical_cols]):
            ax = sns.scatterplot(data=data, x=data[col], y='job_satisfaction_rate', ax = axes[i], hue=cl)
            #ax.axhline(6000, color='r')
        fig.tight_layout()
        plt.title('Диаграммы рассения столбца "job_satisfaction_rate" и количественных столбцов в связке с показателем столбца ' + str(categorical_cols), loc='center')
        plt.show()

In [ ]:
# Отрисовка диаграмм рассеяния в связке с различными типами признаков
print('Анализ диаграмм рассеяния тренировочных данных: ')
draw_scatter_matrix(train_data, ['salary'], cat_col)
draw_scatter_matrix(train_data, ['salary'], bin_col)
draw_scatter_matrix(train_data, ['salary'], discret_col)

In [ ]:
# Отрисовка диаграмм рассеяния в связке с различными типами признаков
print('Анализ диаграмм рассеяния тестовых данных: ')
draw_scatter_matrix(full_test_data, ['salary'], cat_col)
draw_scatter_matrix(full_test_data, ['salary'], bin_col)
draw_scatter_matrix(full_test_data, ['salary'], discret_col)

In [ ]:
# добавление константы для перехвата
print('Анализ мультиколлинеарности тренировочных данных')
X = add_constant(train_data.drop('job_satisfaction_rate', axis=1))
int_columns = X.select_dtypes(include='number').columns.tolist()
# расчет VIF для каждого предиктора
VIFs = pd.DataFrame()
VIFs['Variable'] = X[int_columns].columns


VIFs["VIF"] = [variance_inflation_factor(X[int_columns].values, i)
                          for i in range(len(X[int_columns].columns))]

print(VIFs)

In [ ]:
# добавление константы для перехвата
print('Анализ мультиколлинеарности тестовых данных')
X = add_constant(full_test_data.drop('job_satisfaction_rate', axis=1))
int_columns = X.select_dtypes(include='number').columns.tolist()
# расчет VIF для каждого предиктора
VIFs = pd.DataFrame()
VIFs['Variable'] = X[int_columns].columns


VIFs["VIF"] = [variance_inflation_factor(X[int_columns].values, i)
                          for i in range(len(X[int_columns].columns))]

print(VIFs)

In [ ]:
# Строим матрицу корреляции признаков тренировочных данных
#train_data.drop(columns=cat_col + ['supervisor_evaluation'],inplace=True)
interval_cols = ['job_satisfaction_rate', 'salary']
phik_overview = phik_matrix(train_data, interval_cols=interval_cols) 

plot_correlation_matrix(
    phik_overview.values,
    x_labels=phik_overview.columns,
    y_labels=phik_overview.index,
    vmin=0, vmax=1, color_map='Reds',
    title=r'Таблица корреляции признаков тренировочных данных',
    fontsize_factor=1.5,
    figsize=(20, 15)
) 
plt.show()

In [ ]:
# Матрица корреляции тестовых данных

interval_cols = ['job_satisfaction_rate', 'salary']
phik_overview = phik_matrix(full_test_data, interval_cols=interval_cols)
plot_correlation_matrix(
    phik_overview.values,
    x_labels=phik_overview.columns,
    y_labels=phik_overview.index,
    vmin=0, vmax=1, color_map='Reds',
    title=r'Таблица корреляции признаков тестовых данных',
    fontsize_factor=1.5,
    figsize=(20, 15)
) 


### Выводы после корреляционного анализа
- Значимой мультиколлинераности признаков не наблюдается;
- Утечки целевого признака нет;
- VIF значения подтвердили отсутствие мультиколлиниарности;
- Диаграмма рассеяния подтвердили значимость влияния оценки работаделя и нарушения трудового договора на уровень удовлетворённости рабочим местом;
- Все признаки в большей или меньшей мере вносят своё влияния на целевой признак;

**Добавленное**

- Значимость признака last_year_promo почти в 2 раза выше в тестовых данных;
- Общие распределения признака last_year_promo в диаграммах рассения схожи в диаграммах рассения. Можно предлположить, что важность изменилась из-за малого числа положительных значений признака в тестовых данных;
- В остальном же сильных различий корреляционных анализ не выявил между тестовыми и тренировочными данными.

Дальше нас ждёт обучение разных моделей машинного обучения и отбор лучших признаков.

## **5. Использование пайплайнов и обучение моделей**

In [ ]:
train_data.columns

### 5.1. Пайплайны кодирования

In [ ]:
# Выводу уникальных значений категориальных признаков
for col in cat_col:
    print(test_data[col].unique())

In [ ]:
# создаём пайплайн для подготовки признаков из списка ohe_columns: заполнение пропусков и OHE-кодирование
# SimpleImputer + OHE
ohe_pipe = Pipeline(
    [('simpleImputer_before_ohe', SimpleImputer(missing_values=np.nan, strategy='most_frequent')),
     ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
    ]
    )

In [ ]:
# создаём пайплайн для подготовки признаков из списка ord_columns: заполнение пропусков и Ordinal-кодирование
# SimpleImputer + OE
ord_pipe = Pipeline(
    [('simpleImputer_before_ord', SimpleImputer(missing_values=np.nan, strategy='most_frequent')),
     ('ord',  OrdinalEncoder(
                 categories=[
                    ['junior', 'middle', 'senior'], 
                    ['low', 'medium', 'high'],
                ],
                handle_unknown='use_encoded_value', unknown_value=np.nan
            )
        ),
     ('simpleImputer_after_ord', SimpleImputer(missing_values=np.nan, strategy='most_frequent'))
    ]
)

In [ ]:
# Функция метрики
def smape(y_true, y_pred):
    return np.sum( 2*(np.abs(y_true - y_pred) / ( np.abs(y_true) + np.abs(y_pred) ) ) )*(100/len(y_pred))

In [ ]:
smape_score = make_scorer(smape, greater_is_better=False)

### 5.2. Обучение моделей

In [ ]:
# Готовим признаки к дальнейшей кодировке
ohe_col = bin_col + ['dept']
ord_col = cat_col
ord_col.remove('dept')
nums_col = num_col + discret_col
#nums_col.remove('job_satisfaction_rate')
print(ord_col, nums_col)

In [ ]:
ord_col

In [ ]:
nums_col

In [ ]:
ohe_col

In [ ]:
# создаём общий пайплайн для подготовки данных
data_preprocessor = ColumnTransformer(
    [('ohe', ohe_pipe, ohe_col),
     ('ord', ord_pipe, ord_col),
     ('num', StandardScaler(), nums_col)
    ], 
    remainder='passthrough'
)

In [ ]:
# Финальный пайплайн
pipe_final = Pipeline([
    ('preprocessor', data_preprocessor),
    ('models', DecisionTreeRegressor(random_state=RANDOM_STATE))
])

param_grid = [
    # словарь для модели DecisionTreeClassifier()
    {
        'models': [DecisionTreeRegressor(random_state=RANDOM_STATE)],
        'models__max_depth': range(2,10),
        'models__max_features': range(2,12+1),
        'models__min_samples_split': range(2,10),
        'models__min_samples_leaf':range(2,10),
        'preprocessor__num': [StandardScaler(), MinMaxScaler(), RobustScaler(), 'passthrough']  
    },
    
    {
        'models': [SVR(gamma='auto')],
        'models__C': [0.1,1,5,10,15,20,35,50,100],
       # 'models__gamma': [0.1,1,10,100],
        
        'preprocessor__num': [StandardScaler(), MinMaxScaler(),RobustScaler(), 'passthrough']   
    },

    # словарь для модели LinearRegression()
    {
        'models': [LinearRegression()],
        'preprocessor__num': [StandardScaler(), MinMaxScaler(),RobustScaler(), 'passthrough']  
    }
]

In [ ]:
# Пайплайны для OptunaSearch
pipe_DTR = Pipeline([
    ('preprocessor', data_preprocessor),
    ('models', DecisionTreeRegressor(random_state=RANDOM_STATE))
])
pipe_SVR = Pipeline([
    ('preprocessor', data_preprocessor),
    ('models', SVR(gamma='auto'))
])

In [ ]:
param_DTR= {
    'models__max_depth': distributions.IntDistribution(1, 15),
    'models__min_samples_split': distributions.IntDistribution(2, 10),
    'models__max_features': distributions.IntDistribution(2, 10),
    'models__min_samples_leaf':distributions.IntDistribution(2, 10)
} 

In [ ]:
param_SVR = {
    'models__C': distributions.FloatDistribution(1e-10, 10, log=True),
    'models__gamma': distributions.FloatDistribution(1e-10, 10, log=True)
}

In [ ]:
X_train = train_data.drop(columns='job_satisfaction_rate')
y_train = train_data['job_satisfaction_rate']

In [ ]:
# Поиск лучших гиперпараметров для модели SVR
OS_SVR = OptunaSearchCV(
    pipe_SVR, 
    param_SVR, 
    cv=5,
    scoring= make_scorer(smape, greater_is_better=False),
    random_state=RANDOM_STATE,
    n_trials=15,
   # n_jobs=-1
)
OS_SVR.fit(X_train, y_train)
print(OS_SVR.trials_dataframe().sort_values(by='value', ascending=False).iloc[0] )

In [ ]:
# Поиск лучших гиперпараметров для модели DTR
OS_DTR = OptunaSearchCV(
    pipe_DTR, 
    param_DTR, 
    cv=5,
    scoring= make_scorer(smape, greater_is_better=False),
    random_state=RANDOM_STATE,
    n_trials=100,
   # n_jobs=-1
)
OS_DTR.fit(X_train, y_train)
print(OS_DTR.trials_dataframe().sort_values(by='value', ascending=False).iloc[0] )

In [ ]:
# Поиск лучшей модели и гиперпараметров случайным поиском
randomized_search = RandomizedSearchCV(
    pipe_final,
    param_grid,
    cv=5,
    scoring= make_scorer(smape, greater_is_better=False),
    random_state=RANDOM_STATE,
    n_iter=100,
    n_jobs=-1
)
randomized_search.fit(X_train, y_train)

In [ ]:
# Вывод лучшим моделей и их метрики с параметрами
result = pd.DataFrame(randomized_search.cv_results_)
print(result[
    ['rank_test_score', 'param_models', 'mean_test_score','params']
].sort_values('rank_test_score').head(10))

### 5.3. Определение лучшей модели и метрики

In [ ]:
result_models = []
print('Метрика лучшей модели OptunaSearch_SVR =', -OS_SVR.best_score_)
result_models.append([-OS_SVR.best_score_, OS_SVR.best_estimator_])

print('Метрика лучшей модели OptunaSearch_DTR =', -OS_DTR.best_score_)
result_models.append([-OS_DTR.best_score_, OS_DTR.best_estimator_])

print('Метрика лучшей модели RandomizedSearch между моделями DTR,SVR,LR =', -randomized_search.best_score_)
result_models.append([-randomized_search.best_score_, randomized_search.best_estimator_])

In [ ]:
print('Лучшая модель и её параметры:\n\n', min(result_models)[1])
print ('Метрика лучшей модели на кросс-валидации:', min(result_models)[0])

### Выводы после обучения моделей
- Были реализованы пайплайны кодирования;
- Использован OptunaSearchCV для моделей SVR и DTR, однако существенных результатов это не дало;
- Лучший результат даёт Дерево Решений с высоким показателем на кроссвалидационной выборке;
- Однако достичь необходимого результата не получилось.

P.s. явно что-то делал не так. Пробовал и удалять признаки(например dept). Нужного значения метрики так и не получил. Думаю, что тестовая выборка слишком велика. Можно попробовать увеличить тренировочную синтетическими данными. Дополнительно сильно смущает ситуация, когда Случайный поиск даёт результат лучше, чем OptunaSearch

## **6. Анализ важности признаков**

In [ ]:
pipe_full_final = Pipeline([
    ('preprocessor', data_preprocessor),
    ('selector', SelectKBest(f_classif, k=2)),
    ('models', min(result_models)[1]['models'])
])
param_selector= {
    'selector__k': range(2,len(min(result_models)[1]['preprocessor'].get_feature_names_out())+1),
    'preprocessor__num':[StandardScaler(), MinMaxScaler(), RobustScaler(), 'passthrough'] 
} 
select = GridSearchCV(
    pipe_full_final, 
    param_selector, 
    cv=5,
    scoring= make_scorer(smape, greater_is_better=False),
    n_jobs=-1
)
select.fit(X_train, y_train)

In [ ]:
print('Лучшая модель и её параметры:\n\n', select.best_estimator_)
print ('Метрика лучшей модели на кросс-валидации:', -select.best_score_)

In [ ]:
if len(select.best_estimator_['selector'].get_feature_names_out()) < len(min(result_models)[1]['preprocessor'].get_feature_names_out()):
    print("Признаков стало меньше! Их число равно - ", len(select.best_estimator_['selector'].get_feature_names_out()))
else:
    print('Число признаков осталось без изменений. Их число равно - ', len(select.best_estimator_['selector'].get_feature_names_out()))

In [ ]:
y_test = full_test_data['job_satisfaction_rate']
X_test = full_test_data.drop(columns='job_satisfaction_rate')

y_best_test = smape_score(select.best_estimator_, X_test, y_test)
print('Результат лучшей модели на тестовой выборке', -y_best_test)

In [ ]:

dummy_regr = DummyRegressor(strategy="mean")

dummy_regr.fit(X_train, y_train)

y_pred_dummy = dummy_regr.predict(test_data)

metric_test_dummy = smape_score(dummy_regr, test_data, y_test_data['job_satisfaction_rate'])
print('Результат лучшей модели на тестовой выборке', -metric_test_dummy)


In [ ]:
# Извлекаем лучшую модель и препроцессор
best_model = select.best_estimator_.named_steps['models']
preprocessor = select.best_estimator_.named_steps['preprocessor']
selector = select.best_estimator_.named_steps['selector']
# Преобразуем данные с помощью препроцессора
X_train_transformed = preprocessor.transform(X_train)
name_columns = preprocessor.get_feature_names_out()

X_train_transformed = pd.DataFrame(X_train_transformed, columns=name_columns)


features_names = X_train_transformed.columns[selector.get_support(indices=True)]

X_train_transformed = X_train_transformed[list(features_names)]
# Создаём объяснитель SHAP для линейной модели
explainer = shap.TreeExplainer(best_model, X_train_transformed, feature_perturbation="interventional")

# Рассчитываем значения SHAP для обучающих данных
shap_values = explainer.shap_values(X_train_transformed,check_additivity=False)

# Получаем имена признаков после преобразования
feature_names = preprocessor.get_feature_names_out()
plt.rcParams['axes.titlesize'] = 22
plt.rcParams['axes.labelsize'] = 15
plt.title('График анализа важности признаков')
plt.ylabel('Признаки модели')

# Визуализируем вклад признаков
shap.summary_plot(
    shap_values, 
    X_train_transformed, 
    plot_type="dot", 
    max_display=30, 
    plot_size=(15, 15)  
)



### Выводы после анализа важности признаков
- Как показывала матрица корреляции, а также отбор по лучшим признакам - отдел, в которо работает сотрудник не очень-то и значим;
- Высокие значения зарплаты склоняют модель к тому, что он будет более удовлетворён рабочим местом;
- Как и опыт работы  в компании - чем он больше, тем больше вклад в увеличенное значением модели;
- Наибольший вклад в модель вносит оценка руководством сотрудника - чем выше оценка, тем выше удовлетворённость сотрудика рабочим местом.

## **7. Выводы по выполнению задания 1**

- Познакомились с данными и провели её предобработку;
- Убрали опечатку и неявные попуски;
- Провели исследовательский анализ данных и определили основные взаимосвязи;
- Построили матрицу корреляции признаков, выполнили проверку на мультиколлинеарность;
- Используя пайплайны провели предподготовку данных к обучению моделей;
- Использовали 3 модели МО, а также 2 способа подбора гиперпараметров;
- Провели отбор лучших признаков, а также анализ их важности и вклада в результат модели.

Лучшая модель на текущий момент не дотягивает до необходимого уровня.

# **Задание 2: предсказание увольнения сотрудника из компании**

## **1. Загрузка данных и знакомство с ними**

### 1.1. Загрузка и изучение тренировочных данных

In [ ]:
X_train_z2 = pd.read_csv('https://code.s3.yandex.net/datasets/train_quit.csv',index_col='id')

In [ ]:
ProfileReport(X_train_z2, title="Profiling Report") # Получение основной информации о сведениях в тренировочных данных

In [ ]:
X_train_z2.info()

#### Выводы после знакомства с тренировочными данными
- Данные имеют идентичное содержание и схожее распределение с тренировочными данными задания 1;
- Отсутствуют пропуски в данных;
- Имеется опечатка в столбце level;
- Тип данных соответствует содержимому столбцов;
- Столбец id можно сделать индексом.

### 1.2. Загрузка и изучение тестовых данных

In [ ]:
test_data_z2 = pd.read_csv('https://code.s3.yandex.net/datasets/test_features.csv',index_col='id')
y_test_z2 = pd.read_csv('https://code.s3.yandex.net/datasets/test_target_quit.csv',index_col='id')

In [ ]:
ProfileReport(test_data_z2, title='Profiling Report')

In [ ]:
ProfileReport(y_test_z2, title='Profiling Report')

#### Выводы после знакомства с тестовыми данными
- Поскольку тестовые данные совпадают с данными задания 1, можно взять обработанную тестовую выборку прошлого задания;
- Тестовых данных по прежнему много в сравнении с объёмом тренировочной выборке, а потому можно использовать разбиение тестовых данных на валидационную и тестовые выборки;

### Общие выводы после знакомства с данными
- Пропорции признаков тренировочных данных соответствуют тренировочным данным первого задания;
- В тренировочной выборке отсутствуют пропуски, только опечатка;
- Тестовые данные можно взять и задания 1, поскольку сведения идентичны и там данные уже обработаны;
- Тестовые данные предлагается разделить на валидационную и тестовую выборки.

## **2. Предобработка данных**

In [ ]:
X_train_z2['level'] = X_train_z2['level'].str.replace('sinior','senior') # Избавление от опечатки

In [ ]:
test_data_z2['level'] = test_data_z2['level'].str.replace('sinior','senior') # Избавление от опечатки в тестовых данных

In [ ]:
test_data_z2['workload'] = test_data_z2['workload'].replace(' ', np.nan) # Избавимся от неизвестной нагрузки
test_data_z2['dept'] = test_data_z2['dept'].replace(' ', np.nan)

In [ ]:
# Удаляем дубликаты
X_train_z2.drop_duplicates(inplace=True)

### Выводы после проведения предобработки данных
- Избавились от опечатки в данных;
- Перевели id в с индекс;
- Задали валидационную и тестовую выборки из тестовых данных.

Далее нас ждёт исследовательских анализ новых тренировочных данных.

## **3. Исследовательский анализ данных**

In [ ]:
# Разбиение признаков по типу данных
num_col = ['salary']
discret_col = ['employment_years','supervisor_evaluation']
bin_col = ['last_year_promo', 'last_year_violations','quit']
cat_col = ['dept','level','workload']

### 3.1. Первичный анализ признаков

In [ ]:
draw_boxplot_columns(X_train_z2, num_col) # Ящик с усами непрерывных признаков

In [ ]:
draw_pie_columns(X_train_z2, cat_col) # Информация о категориальных столбцах в виде круговой диаграммы

In [ ]:
draw_pie_columns(X_train_z2, bin_col) # Анализ распределения бинарных данных

In [ ]:
draw_count_columns(X_train_z2, discret_col)

#### Выводы после первичного анализа признаков:
- Средняя зарплата находится на уровне 33,5 тысяч;
- Самый популярный отдел - отдел продаж;
- Больше всего сотрудников уровня Middle;
- Половина всех сотрудников имеют средний уровень загруженности;
- 4% от всех сотрудников получали повышение;
- Почти 18.5% нарушали трудовой договор;
- 25% сотрудников уволились;
- Больше всего в компании молодых сотрудников, проработавших в компании не более 3х лет;
- Чаще всего сотрудники получали оценку работы 3-4;

### 3.2. Анализ влияния категориальных показателей на данные

In [ ]:
draw_pie_columns(X_train_z2, cat_col)

In [ ]:
for col in cat_col:
    draw_pie_columns(X_train_z2.dropna(), bin_col, col)
    draw_histnorm_columns(X_train_z2.dropna(),num_col, col)
    draw_boxplot_columns(X_train_z2.dropna(),num_col, col)
    draw_pie_columns(X_train_z2.dropna(), cat_col, col)
    draw_count_columns(X_train_z2.dropna(), discret_col, col)

#### Выводы после оценки влияния категориальных показателей

Поскольку распределение данных тренировочных данных прошлого и текущего заданий совпадает, основные выводы будут совпадать.
- Больше всего сотрудников из отдела продаж;
- Сотрудников уровня junior около половины от общего числа;
- Порядка 50% сотрудников ощущают средний уровень загруженности;
- Пропорции повышения по отделам сохранены. Не наблюдается такого, что был бы перевес в каком-то отделе относительно числа сотрудников;
- Также сохранены пропорции нарушений трудового договора. Во всех отделах примерно пропорциональное число нарушений;
- Выше всего зарплаты у технологического отдела. Нижний порог у отдела продаж;
- Распределение сотрудников по отделам в целом соответствует уровню занимаемой должности. Нет пропорционального диссбаланса. В отделе маркетинга больше начинающих сотрудников;
- Повышение за последний год получали лишь сотрудники уровня middle и senior. Причём лишь 10% сотрудников уровня senior повышения не получили;
- Сотрудники всех уровней примерно одинаковое количество раз нарушали трудовой договор;
- **Примерно половина junior-сотрудников уволились;**
- Уровень занимаемой должности пропорционален уровню заработной платы, что говорит о наличии корреляции между признаками;
- Большинство нарушений договора допустили работники с низкой нагруженностью;
- Порядка 57% сотрудников с низкой загруженностью уволились.

### 3.3. Анализ влияния бинарных показателей на данные

In [ ]:
draw_pie_columns(X_train_z2, bin_col)

In [ ]:
for col in bin_col:
    draw_pie_columns(X_train_z2.dropna(), bin_col, col)
    draw_histnorm_columns(X_train_z2.dropna(),num_col, col)
    draw_boxplot_columns(X_train_z2.dropna(),num_col, col)
    draw_pie_columns(X_train_z2.dropna(), cat_col, col)
    draw_pie_columns(X_train_z2.dropna(), discret_col, col)

#### Выводы после оценки влияния бинарных признаков

- Тех, кто получил повышение примерно 3% от общего числа сотрудников;
- Уволившихся около 30%;
- Те, кто получили повышение в 90% случаев нарушили трудовой договор;
- Из числа тех, кто уволился лишь 1 получил повышение;
- Сотрудники уровня junior не получали повышения за последний год;
- Имеющие среднюю загруженность получали повышение чаще;
- Яркой зависимости повышения от длительности работы в компании не наблюдается;
- Около половины получивших повышение получили оценку от руководства - 1;
- Примерно 60% нарушивших трудовой договор - уволились;
- Уровень зарплаты не влияет на наличие нарушений трудового договора;
- Уровень сотрудника не связан с наличием нарушения трудового договора;
- Чаще всего трудовой договор нарушали люди умеющии средний уровень загруженности;
- Люди имеющие нарушения трудового договора примущественно получили оценку от руководства - 3;
- Уровень зарплаты уволившихся сотрудников в среднем был ниже.

### 3.4. Анализ влияния дискретных показателей на данные

In [ ]:
for col in discret_col:
    draw_count_columns(X_train_z2.dropna(), bin_col, col)
    draw_histnorm_columns(X_train_z2.dropna(),num_col, col)
    draw_boxplot_columns(X_train_z2.dropna(),num_col, col)
    draw_count_columns(X_train_z2.dropna(), cat_col, col)
    draw_count_columns(X_train_z2.dropna(), discret_col, col)

#### Выводы после оценки влияния дискретных дпоказателей на данные

- Опыт работы в компании пропорцинален уровню заработной плате;
- Во всех отделах работаю сотрудники с разным опытом;
- Сотрудники уровня middle и senior в основонм работают в компании достаточно долго;
- Уволившиеся сотрудники преимущественно имели оценку 4;
- Наличие нарушения трудового договора не связано с оценкой руководства;
- Уровень зарплаты не влияет на оценку руководства;
- Загруженность сотрудников, как их опыт работы в компании не связана с оценкой их работы;

### 3.5. Изучение взаимосвязи удовлетворённости работы и увольнения

In [ ]:
data_jsr_q = y_test_data.merge(y_test_z2,left_index=True, right_index=True) # Объдиняем сведения для анализа

In [ ]:
draw_boxplot_columns(data_jsr_q.dropna(), ['job_satisfaction_rate'], 'quit')

**Нулевая гипотеза - увольнение сотрудника не связано с удовлетворённостью им рабочим местом. Среднее выборок разделённых условием увольнения равно меж собой**

**Альтернативная левосторонняя гипотеза - увольнение сотрудника имеет своё влияние. Уволившиеся сотрудники были менее удовлетворены своим рабочим местом**

In [ ]:
quit_yes = data_jsr_q.query('quit == "yes"')
quit_no = data_jsr_q.query('quit == "no"')
print(f'среднее удовлетворённости при условии, что сотрудник уволился - {np.mean(quit_yes["job_satisfaction_rate"])}')
print(f'среднее удовлетворённости при условии, что сотрудник остался - {np.mean(quit_no["job_satisfaction_rate"])}\n')

alpha = 0.05
results = st.ttest_ind(quit_yes["job_satisfaction_rate"], quit_no["job_satisfaction_rate"], alternative='less')

print('p-значение', results.pvalue,'\n')
if results.pvalue < alpha:
    print('Отвергаем нулевую гипотезу. Принимаем альтернативную левостороннюю о том, что удовлетворённость рабочим местом важна при увольнении, и этот уровень ниже у уволившихся сотрудников.')
else:
    print('Принимаем нулевую гипотезу о том, что удовлетворённость рабочим местом не связана с фактом увольнения сотрудника.')

#### Выводы после оценки взаимосвязи удовлетворённости работой и уходом
Налицо явная взаимосвязь признаков, что было подтверждено через ttest о равнестве среднего. Сотрудники менее удовлетворённые своей работой чаще увольнялись.

### Общие выводы исследовательского анализа
- Больше всего сотрудников работает в отделе продаж;
- Сотрудников уровня junior около половины от общего числа;
- Порядка 50% junior-сотрудников уволилось;
- Выше всего зарплаты в технологическом отделе. Ниже всего в отделе продаж;
- Сотрудники уровня junior не получали повышения за последний год;
- Зарплата тех, кто получал повышение, в среднем выше;
- Те, кто получил повышение не увольнялись, за исключением 1 человека;
- Сильной зависимости между длительностью работы в компании и повышения не наблюдается;
- Опыт работы в компании пропорционален уровню заработной платы;
- Наличие нарушения трудового договора не связано с оценкой руководства;
- Загруженность сотрудников, как и их опыт работы в компании, не связана с оценкой их работы.

Многие из промежуточных выводов должны подтвердиться при проведении корреляционного анализа. Далее этим и займёмся.

### 3.6. Анализ данных уволишивхся сотрудников

In [ ]:
quit_yes_data = X_train_z2.query('quit == "yes"')

In [ ]:
draw_boxplot_columns(quit_yes_data, num_col) # Ящик с усами непрерывных признаков

In [ ]:
draw_pie_columns(quit_yes_data, cat_col) # Информация о категориальных столбцах в виде круговой диаграммы

In [ ]:
draw_pie_columns(X_train_z2, bin_col) # Анализ распределения бинарных данных

In [ ]:
draw_count_columns(X_train_z2, discret_col)

### Портрет уволившегося сотрудника
- Больше всего уволилось из отдела sales;
- С подавляющей вероятностью уволившийся - junior;
- В 50% случаев имел низкую загруженность;
- Не получал повышения за последний год;
- C, примерно, 60% вероятносью не имел нарушений трудового договора;
- Имел невысокую зарплату;
- С вероятностью 90% не имел высокой загруженности;
- Почти наверняка проработал в компании не дольше 3х лет;
- Имел хорошую оценку от руководства;
- Был не очень удовлетворён рабочим местом.

## **4. Добавление нового признака**

In [ ]:
# Добавляем признаки в выборки
X_train_z2['job_satisfaction_rate'] = select.best_estimator_.predict(X_train_z2)
#X_valid_z2['job_satisfaction_rate'] = randomized_search.best_estimator_.predict(X_valid_z2)
test_data_z2['job_satisfaction_rate'] = select.best_estimator_.predict(test_data_z2)

## **5. Корреляционный анализ данных**

In [ ]:
# Добавляем новый признак в численные признаки
num_col.append('job_satisfaction_rate')

In [ ]:
# Диаграммы рассения признаков с учётом влияния категориальных данных
draw_scatter_matrix(X_train_z2, ['salary'], cat_col)
draw_scatter_matrix(X_train_z2, ['salary'], bin_col)
draw_scatter_matrix(X_train_z2, ['salary'], discret_col)

In [ ]:
# Диаграммы рассения признаков с учётом влияния категориальных данных на тестовых данных
full_test_data_z2 = test_data_z2.merge(y_test_z2, left_index=True, right_index=True)
draw_scatter_matrix(full_test_data_z2, ['salary'], cat_col)
draw_scatter_matrix(full_test_data_z2, ['salary'], bin_col)
draw_scatter_matrix(full_test_data_z2, ['salary'], discret_col)

In [ ]:

# добавление константы для перехвата
X = add_constant(X_train_z2.drop('quit', axis=1))
int_columns = X.select_dtypes(include='number').columns.tolist()
# расчет VIF для каждого предиктора
VIFs = pd.DataFrame()
VIFs['Variable'] = X[int_columns].columns

# calculating VIF for each feature
VIFs["VIF"] = [variance_inflation_factor(X[int_columns].values, i)
                          for i in range(len(X[int_columns].columns))]

print(VIFs)

In [ ]:
# Строим матрицу корреляции признаков
#train_data.drop(columns=cat_col+['salary','last_year_promo'],inplace=True)
interval_cols = ['job_satisfaction_rate', 'salary']
phik_overview = phik_matrix(X_train_z2, interval_cols=interval_cols) 

plot_correlation_matrix(
    phik_overview.values,
    x_labels=phik_overview.columns,
    y_labels=phik_overview.index,
    vmin=0, vmax=1, color_map='Reds',
    title=r'Таблица корреляции признаков тренировочных данных',
    fontsize_factor=1.5,
    figsize=(20, 15)
) 


In [ ]:
interval_cols = ['job_satisfaction_rate', 'salary']
phik_overview = phik_matrix(full_test_data_z2, interval_cols=interval_cols) 

plot_correlation_matrix(
    phik_overview.values,
    x_labels=phik_overview.columns,
    y_labels=phik_overview.index,
    vmin=0, vmax=1, color_map='Reds',
    title=r'Таблица корреляции признаков тестовых данных',
    fontsize_factor=1.5,
    figsize=(20, 15)
) 

### Выводы после корреляционного анализа
- Значимой мультиколлинераности признаков не наблюдается;
- Утечки целевого признака нет;
- VIF значения подтвердили отсутствие мультиколлиниарности;
- Диаграмма рассеяния подтвердила значимость зарплаты, опыта работы и других признаков; 
- Все признаки в большей или меньшей мере вносят своё влияния на целевой признак.

Дальше нас ждёт обучение разных моделей машинного обучения и отбор лучших признаков.

- Можем заметить, что в тестовых данных целевой признак иначе коррелирует с некоторыми входными признаками. Это может быть связано с сильными влиянием добавленного признака.

## **6. Использование пайплайнов и обучение моделей**

### 6.1. Пайплайны кодирование и обучение моделей

In [ ]:
ohe_col_z2 = bin_col + ['dept']
ohe_col_z2.remove('quit')
ord_col_z2 = cat_col
ord_col_z2.remove('dept')
#ord_col.remove('level')
nums_col_z2 = num_col + discret_col
nums_col_z2
print(ord_col, nums_col)

In [ ]:
nums_col_z2

In [ ]:
# создаём общий пайплайн для подготовки данных
data_preprocessor = ColumnTransformer(
    [('ohe', ohe_pipe, ohe_col_z2),
     ('ord', ord_pipe, ord_col_z2),
     ('num', StandardScaler(), nums_col_z2)
    ], 
    remainder='passthrough'
)

In [ ]:
pipe_final = Pipeline([
    ('preprocessor', data_preprocessor),
    ('models', DecisionTreeRegressor(random_state=RANDOM_STATE))
])

param_grid = [
    # словарь для модели DecisionTreeClassifier()
    {
        'models': [DecisionTreeClassifier(random_state=RANDOM_STATE)],
        'models__max_depth': range(2,10),
        'models__max_features': range(2,12+1),
        'models__min_samples_split': range(2,10),
        'models__min_samples_leaf':range(2,10),
        'preprocessor__num': [StandardScaler(), MinMaxScaler(), RobustScaler(), 'passthrough']  
    },
    
    {
        'models': [SVC(gamma='auto', class_weight='balanced', random_state=RANDOM_STATE)],
        'models__C': [0.1,1,5,10,15,20,35,50,100],
       # 'models__gamma': [0.1,1,10,100],
        
        'preprocessor__num': [StandardScaler(), MinMaxScaler(),RobustScaler(), 'passthrough']   
    },
    # словарь для модели KNeighborsClassifier() 
    {
        'models': [KNeighborsClassifier()],
        'models__n_neighbors': range(2,20+1),
        'preprocessor__num': [StandardScaler(), MinMaxScaler(),RobustScaler(), 'passthrough']   
    },

    # словарь для модели LinearRegression()
    {
        'models': [LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE)],
        'preprocessor__num': [StandardScaler(), MinMaxScaler(),RobustScaler(), 'passthrough']  
    }
]

In [ ]:
# Задаём выборку с целевым признаков
y_train_z2 = X_train_z2['quit']
X_train_z2 = X_train_z2.drop(columns='quit')


In [ ]:
randomized_search_z2 = RandomizedSearchCV(
    pipe_final,
    param_grid,
    cv=5,
    scoring= 'roc_auc',
    random_state=RANDOM_STATE,
    n_iter=100,
    n_jobs=-1
)
randomized_search_z2.fit(X_train_z2, y_train_z2)

### 6.2. Определение лучшей модели и метрики

In [ ]:
# Вывод лучшим моделей и их метрики с параметрами
result = pd.DataFrame(randomized_search_z2.cv_results_)
print(result[
    ['rank_test_score', 'param_models', 'mean_test_score','params']
].sort_values('rank_test_score').head(10))

In [ ]:
print('Лучшая модель и её параметры:\n\n', randomized_search_z2.best_estimator_)
print ('Метрика лучшей модели на кроссвалидации:', randomized_search_z2.best_score_)
preprocessor = randomized_search_z2.best_estimator_.named_steps['preprocessor']

### Выводы после обучения моделей
- Были реализованы пайплайны кодирования;
- Импользованы модели DTC, SVC, LogisticClassifier, kNN;
- Лучший результат даёт Дерево Решений с высоким показателем на кроссвалидационной выборке;

## **7. Анализ важности признаков**

In [ ]:
pipe_full_final_z2 = Pipeline([
    ('preprocessor', data_preprocessor),
    ('selector', SelectKBest(mutual_info_classif, k=2)),
    ('models',randomized_search_z2.best_estimator_['models'])
])
param_selector_z2 = {
    'selector__k': range(1,len(randomized_search_z2.best_estimator_['preprocessor'].get_feature_names_out())+1),
    'preprocessor__num':[StandardScaler(), MinMaxScaler(), RobustScaler(), 'passthrough'] 
} 
select_z2 = GridSearchCV(
    pipe_full_final_z2, 
    param_selector_z2, 
    cv=5,
    scoring= 'roc_auc',
    n_jobs=-1
)
select_z2.fit(X_train_z2, y_train_z2)
print('Лучшая модель после отбора признаков:', select_z2.best_estimator_)
print ('Метрика лучшей модели на кросс-валидации:', select_z2.best_score_)

In [ ]:
if len(select_z2.best_estimator_['selector'].get_feature_names_out()) < len(randomized_search_z2.best_estimator_['preprocessor'].get_feature_names_out()):
    print("Признаков стало меньше! Их число равно - ", len(select_z2.best_estimator_['selector'].get_feature_names_out()))
else:
    print('Число признаков осталось без изменений. Их число равно - ', len(select_z2.best_estimator_['preprocessor'].get_feature_names_out()))

In [ ]:
y_test_z2 = full_test_data_z2['quit']
X_test_z2 = full_test_data_z2.drop('quit',axis=1)
y_pred = select_z2.predict_proba(X_test_z2)[:,1]
metric_best_model = roc_auc_score(y_test_z2, y_pred)

print('Результат лучшей модели c учётом отбора признаков на тестовой выборке:', metric_best_model)

In [ ]:

dummy_class = DummyClassifier(strategy="most_frequent")

dummy_class.fit(X_train_z2, y_train_z2)

y_pred_dummy = dummy_class.predict_proba(test_data_z2)[:,1]

metric_test_dummy = roc_auc_score(y_test_z2, y_pred_dummy)

print('Результат модели на тестовой выборке', metric_test_dummy)


In [ ]:

# Извлекаем лучшую модель и препроцессор
best_select_model = select_z2.best_estimator_.named_steps['models']
preprocessor_z2 = select_z2.best_estimator_.named_steps['preprocessor']
selector_z2 = select_z2.best_estimator_.named_steps['selector']

# Преобразуем данные с помощью препроцессора
X_train_transformed = preprocessor_z2.transform(X_train_z2)
name_columns = preprocessor_z2.get_feature_names_out()
#print(name_columns)
X_train_transformed = pd.DataFrame(X_train_transformed, columns=name_columns)

features_names = X_train_transformed.columns[selector_z2.get_support(indices=True)]

X_train_transformed = X_train_transformed[list(features_names)]

# Создаём объяснитель SHAP для линейной модели
explainer = shap.TreeExplainer(best_select_model, X_train_transformed, feature_perturbation="interventional")

# Рассчитываем значения SHAP для обучающих данных
shap_values = explainer.shap_values(X_train_transformed)
shap_values = shap_values[:,:,1]
# Получаем имена признаков после преобразования
feature_names = preprocessor.get_feature_names_out()

plt.rcParams['axes.titlesize'] = 22
plt.rcParams['axes.labelsize'] = 15
plt.title('График анализа важности признаков')
plt.ylabel('Признаки модели')
# Визуализируем вклад признаков
shap.summary_plot(
    shap_values, 
    X_train_transformed, 
    plot_type="dot", 
    max_display=30, 
    plot_size=(15, 15)  
)

### Выводы после анализа важности признаков
- Наиболее значимыми признаками стали - опыт работы в компании, уровень сотрудника, удовлетворённость рабочим местом и размер зарпаты;
- Чем ниже все эти показатели, тем сильнее модель склоняется к тому, что сотрудник уволится;

## **8. Выводы по выполнению задания 2**
- Провели загрузку и первичное знакомство с данными;
- Выполнили предобработку данных. Избавились от опечатки, пропусков и дубликатов;
- Провели исследовательский анализ данных. Определили значимость нового признака. Сставили портрет уволившегося сотрудника;
- Добавили новый признак и провели комплексный корреляционный анализ данных с изучением диаграмм рассеяния;
- Построили нескольком моделей машинного обучения. Лучшей моделью оказалась модель Деревьев Решений;
- Отобрали лучшие признаки без потери качества метрики. Отобразили график важности признаков и провели их общий анализ;

# **Общий вывод по работе**

Перед нами стояло 2 задачи: разработка модели предсказывающей уровень удовлетворённости рабочим местом, и создание модели определяющей уволится ли сотрудник.

**Вывод в общем плане**:

**На текущий момент обе задачи так и не были решены**, однако опишем какая работа была выполнена:

- Познакомились с данными, провели индексацию датафреймов, избавились от опечаток, пропусков и дубликатов;
- Провели комплексный исследовательский анализ данных как первичных признаков, так и с оценкой влияния в разрезе;
- Выполненили корреляционный анализ данных, провели тест на мультиколлинеарность. Подтвердили выводы исследовательского анализа;
- Реализовали пайплайны и выполнили обучение моделей с подбором гиперпараметров. Определили лучшую модель;
- Отобрали лучшие признаки и проанализировали их важность;
- Все результаты описали на каждом этапе выполненной работы;

- В ходе выполнения второго задания повторили описанные выше этапы. Также составили общую картину ситуации и портрет уволившегося сотрудника;

На основании этого портрета, а также на изучая важность признаков можем дать общие рекомендации по снижению уровня увольнений в компании.

**Детальный вывод по результатам моделей:**

- В ходе разработки модели предсказывающей уровень удовлетворённости рабочим местом мы использовали 3 модели: DTR, SVR, и LinearRegression. Лучше всего с задачей справлялась модель деревьев решений;
- Использовались 2 способа подбора гиперпараметров - OptunaSearchCV и RandomizedSearchCV;
- Использовались несколько способов обработки численных признаков, а также OHE и OrdE для кодирования категориальных признаков;
- На текущий момент отбор признаков не улучшал показатели метрики, но на тренировочных данных метрика была удовлетворительного значения.

- В ходе разработки модели классификации определяющей уволится ли сотрудника было рассмотрено 4 модели: DTR, SVR, kNN и LogisticRegression. Лучший показатель был достигнут также у дерева решений;
- Были реализованы аналогичные пайплайны подготовки данных;
- Модель деревьев решений показала достойный результат на тренировочных данных.

**Рекомендации по снижению уровня увольнений:**

Глядя на портрет уволившегося сотрудника можно сделать следующие выводы:
- Проверить работу отдела sales;
- Провести переоценку уровня загруженности сотрудников;
- Провести анализ взаимодействия с молодыми сотрудниками уровня junior и тех, кто не столь долго работает в компании;
- Признать важность оценки уровня удовлетворённости рабочим местом сотрудником;
- Выполнить перерасчёт заработной планы работников.

Собирая рекомендации в общую картину - в компании явная проблема с молодыми сотрудниками. Они не склонны надолго задерживаться в компании.